# 1. LLM Gateway

**Goal:** Establish the foundation for our agent by connecting to the DataRobot LLM Gateway.

**Key Concept:**
Instead of managing API keys for every provider (Azure, AWS Bedrock, Google Vertex), DataRobot provides a single, unified endpoint. In this notebook, we verify access to nearly 100 different LLMs and select a base model (`azure/gpt-5` or similar) to power our agent's reasoning capabilities.

In [ ]:
# Install required packages
!pip install -r requirements.txt

In [ ]:
import datarobot as dr
from pprint import pprint

# 1. Initialize client
# This uses your local/notebook DataRobot credentials.
dr_client = dr.Client()

# 2. Query the LLM Gateway catalog
response = dr_client.get(url="genai/llmgw/catalog/")

# 3. Extract supported model IDs
# The catalog returns a list of model metadata; we keep just the string identifiers.
data = response.json()["data"]
supported_llms = [llm_model["model"] for llm_model in data]

# 4. Inspect results
print("Number of LLMs supported by LLM Gateway:", len(supported_llms))
pprint(supported_llms)

In [ ]:
import os
from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

# 1. Load configuration
# .env is optional; defaults are provided below.
load_dotenv()

# 2. Configure the model (via DataRobot LLM Gateway)
MODEL_NAME = os.getenv("MODEL_NAME", "azure/gpt-5-2025-08-07")
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        api_key=dr_client.token,
        base_url=dr_client.endpoint + "/genai/llmgw",
    ),
)

# 3. Define the agent
agent = Agent(model=model)

# 4. Execution (sanity-check)
response = await agent.run("What is the capital of France?")
pprint(response.output)